# LangChain L9 — Level 8 — Research and planning
"Compare the delivery fees of our three logistics partners." One tool call cannot answer
that. The agent must search, open several pages, and synthesise. This is where **planning**
appears, and there are three ways to get it:

```text
Implicit (ReAct)        Explicit (planner + executor)       Hybrid
model -> tool -> model  1. produce a plan (structured)      rough plan, then adapt
      -> tool -> model  2. execute step by step             as results come in
      -> answer         3. synthesise                       (what industry mostly does)
```

To keep the section free of extra API keys, the "web" is a small dictionary of pages. The tool
shapes (`web_search`, `fetch_page`) are exactly what a real search integration exposes.

### Step 1 — A tiny web and two research tools

In [ ]:
FAKE_WEB = {
    "https://swiftbite.example/pricing": ("SwiftBite pricing", "SwiftBite charges a flat delivery fee of 4.50 USD per parcel within the city and 9.00 USD for regional deliveries. Same-day delivery costs an extra 3.00 USD."),
    "https://zipmeal.example/pricing":   ("ZipMeal pricing",   "ZipMeal delivery fee is 3.90 USD for parcels under 5 kg and 7.50 USD above. No regional service."),
    "https://dashdine.example/pricing":  ("DashDine pricing",  "DashDine charges 5.20 USD per city delivery; regional deliveries are 8.00 USD; the first 20 parcels each month are free for enterprise accounts."),
    "https://swiftbite.example/about":   ("About SwiftBite",   "SwiftBite was founded in 2019 and operates in 12 cities."),
}

@tool
def web_search(query: str) -> str:
    """Search the web. Returns up to three results as 'title - url - snippet' lines."""
    words = set(re.findall(r"[a-z]+", query.lower()))
    scored = sorted(FAKE_WEB.items(), key=lambda kv: -len(words & set(re.findall(r"[a-z]+", (kv[1][0] + kv[1][1]).lower()))))
    return "\n".join(f"{title} - {url} - {body[:60]}..." for url, (title, body) in scored[:3])

@tool
def fetch_page(url: str) -> str:
    """Fetch the full text of a web page by url."""
    if url not in FAKE_WEB:
        return "error: page_not_found"
    return FAKE_WEB[url][1]

print(web_search.invoke({"query": "delivery fee pricing partners"}))

### Step 2 — Implicit planning: let the loop decide

The plain agent loop already researches: search, read, read, read, synthesise. Watch the
trajectory. Note the two things it does *not* give you: a plan you can show the user before
work starts, and any guarantee that every partner was read.

In [ ]:
researcher = create_agent(
    model=model, tools=[web_search, fetch_page],
    system_prompt="You are a research assistant. Search, then fetch every relevant page before answering. Cite the urls you used.",
)
result = researcher.invoke({"messages": [{"role": "user", "content": "Research and compare the delivery fees of SwiftBite, ZipMeal and DashDine."}]})
show_messages(result["messages"])

### Step 3 — Explicit planning: plan first, then execute each step

A planner produces a structured `ResearchPlan`; an executor agent runs the steps one by one,
each with its own bounded loop; a final call synthesises. More calls, but every stage is
inspectable, resumable and limitable. Real systems mix both: a rough plan, adapted as results arrive.

In [ ]:
class ResearchPlan(BaseModel):
    """A short, ordered plan for a research task."""
    goal: str = Field(description="One-line restatement of the research goal.")
    steps: list[str] = Field(description="3 to 5 concrete steps, each doable with web_search or fetch_page.")

planner = create_agent(model=model, tools=[], system_prompt="You write short research plans.", response_format=ToolStrategy(ResearchPlan))
plan = planner.invoke({"messages": [{"role": "user", "content": "Compare the delivery fees of SwiftBite, ZipMeal and DashDine."}]})["structured_response"]
print("GOAL :", plan.goal)
for i, step in enumerate(plan.steps, 1):
    print(f"  {i}. {step}")

findings = []
for i, step in enumerate(plan.steps, 1):                       # executor: one bounded agent run per step
    out = researcher.invoke({"messages": [{"role": "user", "content": f"Compare the delivery fees of SwiftBite, ZipMeal and DashDine. Do only this step: {step}"}]})
    findings.append(f"Step {i} ({step}): {text_of(out['messages'][-1])[:300]}")

synthesis = model.invoke([SystemMessage("Write a short comparison from the findings. Be factual."), HumanMessage("\n".join(findings))])
print("\nSYNTHESIS:", text_of(synthesis)[:400])

### Recap

- **Problem seen:** multi-step questions need several searches and a synthesis; one call cannot do it.
- **Layer added:** research tools plus two planning styles: the implicit loop and a planner-executor with structured plans.
- **Evidence:** the trajectory shows search -> fetch x3 -> answer; the explicit plan was visible before any work ran.